# 05 – Results Comparison

Side-by-side comparison of Isolation Forest and Autoencoder.

In [ ]:
import sys, os; sys.path.insert(0, os.path.abspath('..'))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.metrics import RocCurveDisplay
from src.utils.preprocessing import generate_synthetic_dataset, preprocess_pipeline, FEATURE_COLUMNS
from src.models.isolation_forest import SatelliteIsolationForest
from src.models.autoencoder import SatelliteAutoencoder
df = generate_synthetic_dataset(n_samples=120_000, seed=42)
feature_cols = [c for c in FEATURE_COLUMNS if c in df.columns]
train, val, test, scaler, _ = preprocess_pipeline(df, feature_cols)
X_train = train[feature_cols].values; X_val = val[feature_cols].values
X_test = test[feature_cols].values; y_test = test['label'].values

In [ ]:
if_model = SatelliteIsolationForest(); if_model.fit(X_train)
if_metrics = if_model.evaluate(X_test, y_test)
ae_model = SatelliteAutoencoder(input_dim=len(feature_cols), epochs=50)
ae_model.fit(X_train, X_val)
ae_metrics = ae_model.evaluate(X_test, y_test)
results = pd.DataFrame({'Isolation Forest': if_metrics, 'Autoencoder': ae_metrics}).T
print(results.round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
RocCurveDisplay.from_predictions(y_test, if_model.anomaly_scores(X_test), ax=ax, name='Isolation Forest')
RocCurveDisplay.from_predictions(y_test, ae_model.anomaly_scores(X_test), ax=ax, name='Autoencoder')
ax.set_title('ROC Curve Comparison')
plt.tight_layout(); plt.savefig('../results/roc_curve.png', dpi=100); plt.show()

In [ ]:
metrics_names = ['precision', 'recall', 'f1', 'roc_auc']
x = np.arange(len(metrics_names)); width = 0.35
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - width/2, [if_metrics[m] for m in metrics_names], width, label='Isolation Forest')
ax.bar(x + width/2, [ae_metrics[m] for m in metrics_names], width, label='Autoencoder')
ax.axhline(0.91, color='red', linestyle='--', lw=1, label='Target F1')
ax.set_xticks(x); ax.set_xticklabels(['Precision', 'Recall', 'F1', 'ROC-AUC'])
ax.set_ylim(0, 1.05); ax.legend()
plt.tight_layout(); plt.savefig('../results/model_comparison.png', dpi=100); plt.show()